In [8]:
import sqlite3
import pandas as pd

Extracts 3 years of CSV logbooks and 2 years of SQLite database tables,
unions them into a single master dataset, and exports it for Power BI.

In [11]:
# Load the 3 historical logbook CSV files (Years 1 to 3)
dfs = []
csv_files = [
    "../Dataset/diwali_sales_year_1.csv",
    "../Dataset/diwali_sales_year_2.csv",
    "../Dataset/diwali_sales_year_3.csv"
]

for filepath in csv_files:
    df = pd.read_csv(filepath)
    print(f"Successfully loaded {filepath} ({len(df):,} records)")
    dfs.append(df)

# Extract digital database tables from SQLite (Years 4 & 5)
conn = sqlite3.connect("../Dataset/fataka_sales.db")
df_y4 = pd.read_sql("SELECT * FROM transactions_year_4", conn)
df_y5 = pd.read_sql("SELECT * FROM transactions_year_5", conn)
conn.close()

print(f"Successfully loaded 'transactions_year_4' from DB ({len(df_y4):,} records)")
print(f"Successfully loaded 'transactions_year_5' from DB ({len(df_y5):,} records)")

dfs.extend([df_y4, df_y5])

Successfully loaded ../Dataset/diwali_sales_year_1.csv (4,000 records)
Successfully loaded ../Dataset/diwali_sales_year_2.csv (4,000 records)
Successfully loaded ../Dataset/diwali_sales_year_3.csv (4,000 records)
Successfully loaded 'transactions_year_4' from DB (4,000 records)
Successfully loaded 'transactions_year_5' from DB (4,000 records)


In [13]:
# Union all DataFrames into a single master dataset
master_df = pd.concat(dfs, ignore_index=True)

# Standardize types and sort by date
master_df["Date"] = pd.to_datetime(master_df["Date"])
master_df = master_df.sort_values("Date").reset_index(drop=True)
master_df["Quantity_Sold"] = master_df["Quantity_Sold"].fillna(0).astype(int)
master_df["Total_Revenue"] = master_df["Total_Revenue"].fillna(0).astype(float)

# Export master dataset for downstream analysis and Power BI
master_df.to_csv("../Dataset/master_fataka_sales.csv", index=False)

print(f"\nTotal Master Records: {len(master_df):,}")
print(f"Combined 5-Year Revenue: ₹{master_df['Total_Revenue'].sum():,.2f}")
master_df.head()


Total Master Records: 20,000
Combined 5-Year Revenue: ₹30,868,900.00


,Transaction_ID,Date,Days_To_Diwali,Product_ID,Product_Name,Category,Unit_Price,Quantity_Sold,Total_Revenue,Data_Source
0,TX100023,2023-09-28,45,P601,Fancy Sky Shot 7-Shot,Aerial Shots,450,3,1350.0,Logbook_Year_1
1,TX100039,2023-09-28,45,P203,Flower Pots Special,Ground Novelties,350,3,1050.0,Logbook_Year_1
2,TX100009,2023-09-28,45,P401,Atom Bomb (Standard),Bombs,100,6,600.0,Logbook_Year_1
3,TX100034,2023-09-28,45,P701,Sparkler Cones,Novelties,300,5,1500.0,Logbook_Year_1
4,TX100007,2023-09-28,45,P702,Magic Pencil,Novelties,160,3,480.0,Logbook_Year_1
